# Refined QGP: Hubble Cooling and Quark Confinement

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gpartin/LFMPublicExperiments/blob/main/notebooks/LFM_QGP_Refined.ipynb)

## What This Notebook Demonstrates

A refined QGP experiment with **Hubble-like damping** to model cosmological cooling:

$$\frac{\partial^2 \Psi}{\partial t^2} + 2\gamma \frac{\partial \Psi}{\partial t} = c^2 \nabla^2 \Psi - \chi^2 \Psi$$

- 15 initial quark blobs cool and confine as $\chi$ recovers
- Critical temperature derived from $\chi_0$: $T_c \propto \sqrt{3/4} \cdot \chi_0 / \sqrt{\kappa}$
- Tracks both confined and deconfined phases

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

chi0, kappa, c = 19.0, 0.016, 1.0
nx = 400
dx, dt = 1.0, 0.01
n_steps = 15000  # Reduced for Colab
x = np.arange(nx) * dx
H_initial = 0.01  # Hubble parameter

# 15 initial quark blobs
np.random.seed(42)
n_blobs = 15
blob_width = nx / n_blobs
E = np.zeros(nx)
for i in range(n_blobs):
    cx = (i + 0.5) * blob_width * dx
    E += 15.0 * np.exp(-((x - cx) / (blob_width * dx * 0.3))**2)
E *= (1 + 0.1 * np.random.randn(nx))
E_prev = E.copy()

order_param = []
chi_min_hist = []
energy_hist = []
confined_frac = []

for step in range(n_steps):
    t_now = step * dt
    gamma = H_initial / (1 + t_now * 0.01)  # Decreasing Hubble
    E_sq = E**2
    chi_sq = chi0**2 - kappa * E_sq
    chi = np.sqrt(np.maximum(chi_sq, 0.01))
    lap = np.zeros(nx)
    lap[1:-1] = (E[:-2] + E[2:] - 2*E[1:-1]) / dx**2
    # Damped GOV-01
    E_new = (2*E - E_prev + dt**2 * (c**2 * lap - chi**2 * E)) / (1 + 2*gamma*dt)
    E_new += 2*gamma*dt / (1 + 2*gamma*dt) * E_prev  # Correct damped update
    # Periodic BC
    E_new[0] = E_new[1]
    E_new[-1] = E_new[-2]
    E_prev, E = E.copy(), E_new.copy()
    phi = np.mean(chi) / chi0
    order_param.append(phi)
    chi_min_hist.append(np.min(chi))
    energy_hist.append(np.mean(E_sq))
    conf = np.sum(chi > 0.5 * chi0) / nx
    confined_frac.append(conf)

order_param = np.array(order_param)
energy_hist = np.array(energy_hist)
confined_frac = np.array(confined_frac)
t_arr = np.arange(n_steps) * dt

# Classify phases
has_deconfined = np.any(np.array(order_param) < 0.5)
has_confined = np.any(np.array(order_param) > 0.8)
chi_recovery = order_param[-1]

print('=' * 60)
print('QGP REFINED (HUBBLE COOLING) RESULTS')
print('=' * 60)
print(f'Initial phi: {order_param[0]:.4f}')
print(f'Final phi:   {order_param[-1]:.4f}')
print(f'Deconfined phase present: {has_deconfined}')
print(f'Confined phase reached:   {has_confined}')
print(f'Chi recovery:             {chi_recovery*100:.1f}%')
print(f'\nT_c proportional to sqrt(3/4) * chi0 / sqrt(kappa):')
print(f'  T_c ~ {np.sqrt(3/4) * chi0 / np.sqrt(kappa):.1f} (natural units)')
print(f'\nH0 REJECTED: {(has_deconfined and has_confined) or chi_recovery > 0.3}')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

ax = axes[0, 0]
ax.plot(t_arr, order_param, 'b-', lw=1.5)
ax.axhline(1.0, color='k', ls=':', alpha=0.3)
ax.axhline(0.5, color='r', ls='--', alpha=0.5, label='Transition threshold')
ax.set_xlabel('Time')
ax.set_ylabel('phi = chi/chi_0')
ax.set_title('Order Parameter (Confinement)')
ax.legend()
ax.grid(alpha=0.3)

ax = axes[0, 1]
ax.semilogy(t_arr, energy_hist, 'r-', lw=1.5)
ax.set_xlabel('Time')
ax.set_ylabel('Mean E^2')
ax.set_title('Energy Density (Hubble cooling)')
ax.grid(alpha=0.3)

ax = axes[1, 0]
ax.plot(t_arr, confined_frac, 'g-', lw=1.5)
ax.set_xlabel('Time')
ax.set_ylabel('Fraction with chi > chi_0/2')
ax.set_title('Confined Volume Fraction')
ax.grid(alpha=0.3)

ax = axes[1, 1]
ax.plot(energy_hist, order_param, 'k-', lw=1)
ax.set_xlabel('Energy density')
ax.set_ylabel('Order parameter phi')
ax.set_title('Phase Diagram')
ax.grid(alpha=0.3)

plt.suptitle('Refined QGP with Hubble Cooling', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Key Result

- Hubble-like damping models cosmological cooling
- Initial hot phase is deconfined ($\chi \ll \chi_0$)
- As system cools, $\chi$ recovers and quarks confine
- The phase diagram shows a clear confinement transition
- Critical temperature scales with $\chi_0$ and $\kappa$ — both LFM parameters